# 01 — Filtrado de datos crudos ENOE

Lee los archivos crudos de `datos/DATOS MASIVOS ENOE/`,
extrae las columnas necesarias y guarda los resultados en `datos/filtrado/`.

- Si un archivo ya fue filtrado, se omite.
- La columna del factor de expansión se normaliza siempre como `fac_tri`,
  sin importar si en el crudo se llama `fac` (archivos hasta 2021-T1)
  o `fac_tri` (archivos de 2021-T2 en adelante, según notas INEGI).
- Solo se guarda el resultado filtrado — los crudos no se mueven.

In [1]:
import pandas as pd
import numpy as np
import os
import re
import gc
import glob
from pathlib import Path

RAIZ          = Path('..').resolve()
RUTA_CRUDO    = RAIZ / 'datos' / 'DATOS MASIVOS ENOE'
RUTA_FILTRADO = RAIZ / 'datos' / 'filtrado'

RUTA_FILTRADO.mkdir(parents=True, exist_ok=True)

# Verificar rutas
print(f'Crudo    : {RUTA_CRUDO}')
print(f'  existe : {RUTA_CRUDO.exists()}')
print(f'Filtrado : {RUTA_FILTRADO}')

crudos = sorted(RUTA_CRUDO.glob('*SDEMT*.csv')) if RUTA_CRUDO.exists() else []
print(f'\nArchivos crudos encontrados: {len(crudos)}')
if len(crudos) == 0:
    print(f'Verifica que la carpeta tenga archivos CSV con "SDEMT" en el nombre.')

Crudo    : /Users/mateos_kvn/HackODS26/LoboEnsambladores/datos/DATOS MASIVOS ENOE
  existe : True
Filtrado : /Users/mateos_kvn/HackODS26/LoboEnsambladores/datos/filtrado

Archivos crudos encontrados: 82


## Catálogos

In [2]:
ENTIDADES = {
    1:'Aguascalientes', 2:'Baja California', 3:'Baja California Sur',
    4:'Campeche', 5:'Coahuila', 6:'Colima', 7:'Chiapas', 8:'Chihuahua',
    9:'Ciudad de México', 10:'Durango', 11:'Guanajuato', 12:'Guerrero',
    13:'Hidalgo', 14:'Jalisco', 15:'México', 16:'Michoacán', 17:'Morelos',
    18:'Nayarit', 19:'Nuevo León', 20:'Oaxaca', 21:'Puebla', 22:'Querétaro',
    23:'Quintana Roo', 24:'San Luis Potosí', 25:'Sinaloa', 26:'Sonora',
    27:'Tabasco', 28:'Tamaulipas', 29:'Tlaxcala', 30:'Veracruz',
    31:'Yucatán', 32:'Zacatecas',
}
SEXO     = {1:'Hombre', 2:'Mujer'}
FORMAL   = {1:'Formal', 2:'Informal'}
NIV_INS  = {1:'Sin instrucción', 2:'Básica', 3:'Media superior', 4:'Superior', 5:'No especificado'}
SECTORES = {
    1:'Agropecuario', 2:'Industria extractiva', 3:'Manufactura',
    4:'Construcción', 5:'Comercio', 6:'Restaurantes y hospedaje',
    7:'Transportes y comunicaciones', 8:'Servicios financieros y profesionales',
    9:'Servicios sociales', 10:'Servicios diversos', 11:'Gobierno',
}
OCUPACION = {
    1:'Funcionarios y directivos', 2:'Profesionistas y técnicos',
    3:'Trabajadores de la educación', 4:'Trabajadores del arte',
    5:'Actividades administrativas', 6:'Comerciantes y vendedores',
    7:'Servicios personales', 8:'Actividades agrícolas',
    9:'Industria extractiva y construcción', 10:'Operadores de maquinaria',
    11:'Otras ocupaciones',
}

## Funciones

In [3]:
def parsear_nombre(ruta):
    """Extrae trimestre y año del nombre. Acepta los tres formatos del INEGI."""
    n = Path(ruta).stem.upper()
    m = re.search(r'SDEMT(\d)(\d{2})$', n)
    if not m:
        return None
    trim, yy = int(m.group(1)), m.group(2)
    anio = int('20' + yy)
    return (trim, anio) if trim in [1,2,3,4] and 2000 <= anio <= 2030 else None


def nombre_salida(ruta):
    info = parsear_nombre(ruta)
    if not info:
        return None
    trim, anio = info
    return f'ENOE_T{trim}{str(anio)[-2:]}.csv'


def a_num(s):
    """Convierte a numérico. Necesario porque algunos archivos 2023+ tienen campos como texto."""
    return pd.to_numeric(s, errors='coerce')


def filtrar(ruta):
    enc  = pd.read_csv(ruta, encoding='latin1', nrows=0)
    cols = list(enc.columns)

    # Detectar nombre de la columna de estado
    # En 2023+ pasó a llamarse cve_ent; antes era ent
    col_est = 'cve_ent' if 'cve_ent' in cols else 'ent'

    # Detectar nombre del factor de expansión
    # Según notas INEGI, el cambio de fac → fac_tri ocurrió en agosto 2021
    col_fac = 'fac_tri' if 'fac_tri' in cols else 'fac'

    quiero = [
        col_est, col_fac,
        'sex', 'eda', 'c_ocu11c', 'rama_est2',
        'niv_ins', 'anios_esc',
        'hrsocup', 'ingocup', 'ing_x_hrs',
        'emp_ppal', 'clase2',
    ]
    cols_leer = [c for c in quiero if c in cols]
    df = pd.read_csv(ruta, encoding='latin1', low_memory=False, usecols=cols_leer)

    # Solo población ocupada
    if 'clase2' in df.columns:
        df = df[a_num(df['clase2']) == 1].copy()
        df.drop(columns=['clase2'], inplace=True)

    # Normalizar tipos numéricos
    for c in [col_est, col_fac, 'sex', 'c_ocu11c', 'rama_est2',
              'niv_ins', 'emp_ppal', 'hrsocup', 'ingocup']:
        if c in df.columns:
            df[c] = a_num(df[c])

    # Renombrar al nombre canónico (fac_tri siempre, para que 02_calculo no tenga que detectar)
    df.rename(columns={col_est: 'cve_ent', col_fac: 'fac_tri'}, inplace=True, errors='ignore')

    # Factores 0 corresponden a viviendas sin entrevista (corregidos por INEGI en varias versiones)
    if 'fac_tri' in df.columns:
        df['fac_tri'] = df['fac_tri'].where(df['fac_tri'] > 0)

    # Ingreso por hora: usar el campo INEGI si existe; calcularlo desde ingocup/hrsocup si no
    if 'ing_x_hrs' in df.columns:
        df['ingreso_hora'] = a_num(df['ing_x_hrs']).where(a_num(df['ing_x_hrs']) > 0)
    if 'ingocup' in df.columns and 'hrsocup' in df.columns:
        i, h = a_num(df['ingocup']), a_num(df['hrsocup'])
        calc  = (i / (h * 4.33)).round(2)
        if 'ingreso_hora' not in df.columns:
            df['ingreso_hora'] = calc.where((i > 0) & (h > 0))
        else:
            df['ingreso_hora'] = df['ingreso_hora'].fillna(calc.where((i > 0) & (h > 0)))
    df['ingreso_hora'] = df['ingreso_hora'].where(df['ingreso_hora'] > 0)

    # Salario semanal estimado
    if 'ingocup' in df.columns:
        i = a_num(df['ingocup'])
        df['salario_semanal'] = (i / 4.33).round(2).where(i > 0)

    # Etiquetas de texto
    df['entidad']    = df['cve_ent'].map(ENTIDADES)  if 'cve_ent'   in df.columns else np.nan
    df['sexo']       = df['sex'].map(SEXO)            if 'sex'       in df.columns else np.nan
    df['sector']     = df['rama_est2'].map(SECTORES)  if 'rama_est2' in df.columns else np.nan
    df['nivel_educ'] = df['niv_ins'].map(NIV_INS)     if 'niv_ins'   in df.columns else np.nan
    df['ocupacion']  = df['c_ocu11c'].map(OCUPACION)  if 'c_ocu11c'  in df.columns else np.nan
    df['formalidad'] = df['emp_ppal'].map(FORMAL)      if 'emp_ppal'  in df.columns else np.nan

    return df

## Proceso principal

In [4]:
for ruta in crudos:
    nf = nombre_salida(ruta)
    if not nf:
        print(f'  OMITIDO (nombre no reconocido): {ruta.name}')
        continue

    salida = RUTA_FILTRADO / nf
    if salida.exists():
        print(f'  ya existe: {nf}')
        continue

    try:
        df = filtrar(ruta)
        df.to_csv(salida, index=False, encoding='utf-8-sig')
        n_fac = int(df['fac_tri'].notna().sum()) if 'fac_tri' in df.columns else 0
        print(f'  {ruta.name} -> {nf}  ({len(df):,} filas, {n_fac:,} con fac_tri)')
        del df; gc.collect()
    except Exception as e:
        print(f'  ERROR {ruta.name}: {e}')

print('\nFiltrado completado.')

  ENOEN_SDEMT121.csv -> ENOE_T121.csv  (148,991 filas, 148,991 con fac_tri)
  ENOEN_SDEMT122.csv -> ENOE_T122.csv  (177,295 filas, 177,295 con fac_tri)
  ENOEN_SDEMT221.csv -> ENOE_T221.csv  (167,677 filas, 167,677 con fac_tri)
  ENOEN_SDEMT222.csv -> ENOE_T222.csv  (182,163 filas, 182,163 con fac_tri)
  ENOEN_SDEMT320.csv -> ENOE_T320.csv  (120,316 filas, 120,316 con fac_tri)
  ENOEN_SDEMT321.csv -> ENOE_T321.csv  (188,379 filas, 188,379 con fac_tri)
  ENOEN_SDEMT322.csv -> ENOE_T322.csv  (179,396 filas, 179,396 con fac_tri)
  ENOEN_SDEMT420.csv -> ENOE_T420.csv  (152,033 filas, 152,033 con fac_tri)
  ENOEN_SDEMT421.csv -> ENOE_T421.csv  (193,202 filas, 193,202 con fac_tri)
  ENOEN_SDEMT422.csv -> ENOE_T422.csv  (180,687 filas, 180,687 con fac_tri)
  ENOE_SDEMT120.csv -> ENOE_T120.csv  (184,064 filas, 184,064 con fac_tri)
  ENOE_SDEMT123.csv -> ENOE_T123.csv  (204,787 filas, 204,787 con fac_tri)
  ENOE_SDEMT124.csv -> ENOE_T124.csv  (194,261 filas, 194,261 con fac_tri)
  ENOE_SDEMT125

In [5]:
# Resumen
filtrados = sorted(RUTA_FILTRADO.glob('ENOE_T*.csv'))
total_mb  = sum(f.stat().st_size for f in filtrados) / 1_048_576
print(f'Archivos en filtrado/ ({len(filtrados)} total, {total_mb:.0f} MB):\n')
for f in filtrados:
    mb = f.stat().st_size / 1_048_576
    print(f'  {f.name:<26}  {mb:5.1f} MB')

Archivos en filtrado/ (82 total, 1715 MB):

  ENOE_T105.csv                20.1 MB
  ENOE_T106.csv                21.0 MB
  ENOE_T107.csv                21.1 MB
  ENOE_T108.csv                21.1 MB
  ENOE_T109.csv                20.1 MB
  ENOE_T110.csv                20.2 MB
  ENOE_T111.csv                19.8 MB
  ENOE_T112.csv                20.2 MB
  ENOE_T113.csv                19.7 MB
  ENOE_T114.csv                20.2 MB
  ENOE_T115.csv                20.4 MB
  ENOE_T116.csv                20.2 MB
  ENOE_T117.csv                20.1 MB
  ENOE_T118.csv                20.2 MB
  ENOE_T120.csv                22.2 MB
  ENOE_T121.csv                17.9 MB
  ENOE_T122.csv                21.3 MB
  ENOE_T123.csv                24.6 MB
  ENOE_T124.csv                23.3 MB
  ENOE_T125.csv                22.7 MB
  ENOE_T205.csv                20.4 MB
  ENOE_T206.csv                21.1 MB
  ENOE_T207.csv                21.2 MB
  ENOE_T208.csv                21.3 MB
  ENOE_T209.csv     